# YOLO SAM31 P2 Two-Class Precision Fine-Tuning

Run this notebook on the GPU cluster after cloning/pulling the repo. It builds a temporary two-class YOLO dataset (`nucleus` + `clear_cell_boundary`), drops `compact_cell_boundary` and `stroma`, uses mild oversampling, and fine-tunes with boundary-preserving low-augmentation settings to reduce false-positive boundary masks.

In [ ]:
from pathlib import Path
import csv
import json
import os
import shlex
import shutil
import subprocess
import sys
import time
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, clear_output, display


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'training_data/dataset/yolo_seg_dataset').exists():
            return candidate
    raise FileNotFoundError('Could not find training_data/dataset/yolo_seg_dataset from current path')


def has_nvidia_gpu() -> bool:
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=5)
        return result.returncode == 0
    except Exception:
        return False


def choose_training_model(output_root: Path, reference_model_dir: Path, repo_root: Path) -> Path | str:
    candidates = []
    if output_root.exists():
        candidates.extend(sorted(output_root.glob('*/weights/best.pt'), key=lambda p: p.stat().st_mtime, reverse=True))
    candidates.extend([
        reference_model_dir / 'yolo_sam31_p2_24tiles_best.pt',
        reference_model_dir / 'cellseg1_cgh_p2_yolo_best.pt',
        repo_root / 'yolov8s-seg.pt',
    ])
    for candidate in candidates:
        if isinstance(candidate, Path) and candidate.exists():
            return candidate
    return 'yolov8s-seg.pt'


REPO_ROOT = find_repo_root(Path.cwd())
SOURCE_YOLO_DATASET = REPO_ROOT / 'training_data/dataset/yolo_seg_dataset'
OUTPUT_ROOT = REPO_ROOT / 'outputs/yolo_cluster_live'
REFERENCE_MODEL_DIR = REPO_ROOT / 'training_data/reference_models'
EXPERIMENT_NAME = 'yolo_2class_nucleus_clear_boundary_precision'
YOLO_DATASET = OUTPUT_ROOT / 'datasets' / EXPERIMENT_NAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REFERENCE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Keep only nucleus and clear-cell boundary. Drop compact-cell boundary and stroma.
KEEP_CLASS_MAP = {0: 0, 1: 1}
CLASS_NAMES = {0: 'nucleus', 1: 'clear_cell_boundary'}

# Mildly duplicate the hardest dense train tiles. Values are total copies.
# This keeps useful boundary examples without encouraging boundary hallucination.
OVERSAMPLE_MULTIPLIERS = {
    'p2_tile_12': 2,
    'p2_tile_14': 2,
    'p2_tile_16': 2,
}

EPOCHS = 100
IMGSZ = 512
BATCH = 8
WORKERS = 4
PATIENCE = 45
DEVICE = '0' if has_nvidia_gpu() else 'cpu'
MODEL = choose_training_model(OUTPUT_ROOT, REFERENCE_MODEL_DIR, REPO_ROOT)
FINE_TUNING = isinstance(MODEL, Path) and MODEL.exists() and MODEL.name != 'yolov8s-seg.pt'
LR0 = 0.00025 if FINE_TUNING else 0.001
RUN_NAME = 'yolo_2class_precision_' + datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = OUTPUT_ROOT / RUN_NAME
LIVE_LOG = OUTPUT_ROOT / f'{RUN_NAME}_live_training.log'
RUNTIME_DATA_YAML = YOLO_DATASET / 'data.yaml'
PRED_CONF = 0.45
PRED_IOU = 0.40
PRED_MAX_DET = 80

print('REPO_ROOT:', REPO_ROOT)
print('SOURCE_YOLO_DATASET:', SOURCE_YOLO_DATASET)
print('GENERATED_YOLO_DATASET:', YOLO_DATASET)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('MODEL:', MODEL)
print('FINE_TUNING:', FINE_TUNING)
print('LR0:', LR0)
print('DEVICE:', DEVICE)
print('RUN_NAME:', RUN_NAME)
print('PRED_CONF:', PRED_CONF)
print('PRED_IOU:', PRED_IOU)

In [ ]:
try:
    import ultralytics
    print('ultralytics:', ultralytics.__version__)
except Exception as exc:
    raise RuntimeError('Ultralytics is not installed in this environment. Install it once with: python -m pip install ultralytics') from exc


def filtered_label_lines(label_path: Path) -> tuple[list[str], dict[str, int]]:
    lines = []
    counts = {name: 0 for name in CLASS_NAMES.values()}
    for raw in label_path.read_text(encoding='utf-8').splitlines():
        parts = raw.strip().split()
        if not parts:
            continue
        old_cls = int(float(parts[0]))
        if old_cls not in KEEP_CLASS_MAP:
            continue
        new_cls = KEEP_CLASS_MAP[old_cls]
        lines.append(' '.join([str(new_cls), *parts[1:]]))
        counts[CLASS_NAMES[new_cls]] += 1
    return lines, counts


def write_data_yaml(path: Path) -> None:
    names = '\n'.join(f'  {idx}: {name}' for idx, name in CLASS_NAMES.items())
    path.write_text(
        f'path: {YOLO_DATASET}\n'
        'train: images/train\n'
        'val: images/val\n\n'
        f'names:\n{names}\n',
        encoding='utf-8',
    )


def prepare_two_class_dataset() -> pd.DataFrame:
    if YOLO_DATASET.exists():
        shutil.rmtree(YOLO_DATASET)
    for split in ['train', 'val']:
        (YOLO_DATASET / 'images' / split).mkdir(parents=True, exist_ok=True)
        (YOLO_DATASET / 'labels' / split).mkdir(parents=True, exist_ok=True)

    rows = []
    for split in ['train', 'val']:
        for image_path in sorted((SOURCE_YOLO_DATASET / 'images' / split).glob('*.png')):
            tile_id = image_path.stem
            label_path = SOURCE_YOLO_DATASET / 'labels' / split / f'{tile_id}.txt'
            assert label_path.exists(), label_path
            lines, counts = filtered_label_lines(label_path)
            copies = OVERSAMPLE_MULTIPLIERS.get(tile_id, 1) if split == 'train' else 1
            for copy_idx in range(copies):
                suffix = '' if copy_idx == 0 else f'_os{copy_idx + 1}'
                out_stem = f'{tile_id}{suffix}'
                shutil.copy2(image_path, YOLO_DATASET / 'images' / split / f'{out_stem}.png')
                (YOLO_DATASET / 'labels' / split / f'{out_stem}.txt').write_text('\n'.join(lines) + ('\n' if lines else ''), encoding='utf-8')
                rows.append({
                    'tile_id': tile_id,
                    'copy_id': copy_idx + 1,
                    'split': split,
                    **counts,
                    'total': sum(counts.values()),
                })

    write_data_yaml(RUNTIME_DATA_YAML)
    summary = pd.DataFrame(rows)
    summary.to_csv(YOLO_DATASET / 'label_counts_2class_precision.csv', index=False)
    (YOLO_DATASET / 'conversion_summary_2class_precision.json').write_text(json.dumps({
        'source_dataset': str(SOURCE_YOLO_DATASET),
        'output_dataset': str(YOLO_DATASET),
        'classes': CLASS_NAMES,
        'keep_class_map': KEEP_CLASS_MAP,
        'dropped_classes': {'2': 'compact_cell_boundary', '3': 'stroma'},
        'oversample_multipliers': OVERSAMPLE_MULTIPLIERS,
        'rows': rows,
    }, indent=2), encoding='utf-8')
    return summary


summary = prepare_two_class_dataset()
image_files = sorted((YOLO_DATASET / 'images').glob('*/*.png'))
label_files = sorted((YOLO_DATASET / 'labels').glob('*/*.txt'))
assert len(image_files) == len(label_files), (len(image_files), len(label_files))
assert (RUNTIME_DATA_YAML).exists(), RUNTIME_DATA_YAML

print(RUNTIME_DATA_YAML)
print(RUNTIME_DATA_YAML.read_text())
print('Images:', len(image_files), 'Labels:', len(label_files))
display(summary.groupby('split')[['nucleus', 'clear_cell_boundary', 'total']].sum())
display(summary)


In [ ]:
def nvidia_snapshot() -> str:
    try:
        result = subprocess.run([
            'nvidia-smi',
            '--query-gpu=name,memory.used,memory.total,utilization.gpu,temperature.gpu',
            '--format=csv,noheader,nounits',
        ], capture_output=True, text=True, timeout=5)
        return result.stdout.strip() if result.returncode == 0 else result.stderr.strip()
    except Exception as exc:
        return f'nvidia-smi unavailable: {exc}'

def tail_text(path: Path, lines: int = 18) -> str:
    if not path.exists():
        return '(log not created yet)'
    text = path.read_text(errors='replace')
    return '\\n'.join(text.splitlines()[-lines:])

def load_results(run_dir: Path):
    path = run_dir / 'results.csv'
    if not path.exists():
        return None
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    return df

def plot_live_results(df: pd.DataFrame, out_path: Path):
    if df is None or df.empty:
        return
    x = df['epoch'] if 'epoch' in df.columns else range(len(df))
    loss_cols = [c for c in df.columns if 'loss' in c.lower()]
    metric_cols = [c for c in df.columns if c.startswith('metrics/') or 'mAP' in c or 'precision' in c or 'recall' in c]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for col in loss_cols:
        axes[0].plot(x, df[col], label=col.replace('train/', '').replace('val/', 'val '))
    axes[0].set_title('YOLO losses')
    axes[0].set_xlabel('epoch')
    axes[0].grid(alpha=0.25)
    axes[0].legend(fontsize=8)
    for col in metric_cols:
        axes[1].plot(x, df[col], label=col.replace('metrics/', ''))
    axes[1].set_title('Validation metrics')
    axes[1].set_xlabel('epoch')
    axes[1].grid(alpha=0.25)
    axes[1].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(out_path, dpi=160)
    plt.show()

def live_dashboard(run_dir: Path, log_path: Path):
    clear_output(wait=True)
    print('Run:', run_dir)
    print('Time:', datetime.now().isoformat(timespec='seconds'))
    print('GPU:', nvidia_snapshot())
    df = load_results(run_dir)
    if df is not None and not df.empty:
        display(df.tail(5))
        plot_live_results(df, run_dir / 'live_metrics.png')
    else:
        print('Waiting for results.csv...')
    print('--- log tail ---')
    print(tail_text(log_path))

In [ ]:
YOLO_RUNNER = REPO_ROOT / 'training/run_yolo_segment_train.py'
assert YOLO_RUNNER.exists(), YOLO_RUNNER

cmd = [
    sys.executable, str(YOLO_RUNNER),
    f'model={MODEL}',
    f'data={RUNTIME_DATA_YAML}',
    f'epochs={EPOCHS}',
    f'imgsz={IMGSZ}',
    f'batch={BATCH}',
    f'device={DEVICE}',
    f'workers={WORKERS}',
    f'patience={PATIENCE}',
    'optimizer=AdamW',
    f'lr0={LR0}',
    'mosaic=0.0',
    'close_mosaic=0',
    'copy_paste=0.0',
    'degrees=2',
    'translate=0.02',
    'scale=0.10',
    'fliplr=0.5',
    'mixup=0.0',
    'hsv_h=0.01',
    'hsv_s=0.25',
    'hsv_v=0.20',
    'erasing=0.0',
    'overlap_mask=False',
    'mask_ratio=2',
    'cls=1.0',
    f'project={OUTPUT_ROOT}',
    f'name={RUN_NAME}',
]

print('Command:')
print(' '.join(shlex.quote(str(x)) for x in cmd))

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env.setdefault('WANDB_MODE', 'offline')

with LIVE_LOG.open('w', encoding='utf-8') as log_handle:
    proc = subprocess.Popen(cmd, cwd=REPO_ROOT, stdout=log_handle, stderr=subprocess.STDOUT, text=True, env=env)
    while proc.poll() is None:
        live_dashboard(RUN_DIR, LIVE_LOG)
        time.sleep(15)
    return_code = proc.wait()

live_dashboard(RUN_DIR, LIVE_LOG)
if return_code != 0:
    raise RuntimeError(f'YOLO training failed with return code {return_code}. See {LIVE_LOG}')
print('Training complete')

In [ ]:
best_pt = RUN_DIR / 'weights/best.pt'
last_pt = RUN_DIR / 'weights/last.pt'
deployed_pt = REFERENCE_MODEL_DIR / 'yolo_2class_nucleus_clear_boundary_precision_best.pt'
if best_pt.exists():
    shutil.copy2(best_pt, deployed_pt)

summary = {
    'timestamp': datetime.now().isoformat(timespec='seconds'),
    'repo_root': str(REPO_ROOT),
    'source_dataset': str(SOURCE_YOLO_DATASET),
    'training_dataset': str(YOLO_DATASET),
    'runtime_data_yaml': str(RUNTIME_DATA_YAML),
    'run_dir': str(RUN_DIR),
    'live_log': str(LIVE_LOG),
    'best_pt': str(best_pt),
    'last_pt': str(last_pt),
    'deployed_pt': str(deployed_pt) if best_pt.exists() else '',
    'model_start': str(MODEL),
    'fine_tuning': FINE_TUNING,
    'classes': CLASS_NAMES,
    'dropped_classes': {'2': 'compact_cell_boundary', '3': 'stroma'},
    'oversample_multipliers': OVERSAMPLE_MULTIPLIERS,
    'epochs': EPOCHS,
    'imgsz': IMGSZ,
    'batch': BATCH,
    'device': DEVICE,
    'lr0': LR0,
    'prediction_conf': PRED_CONF,
    'prediction_iou': PRED_IOU,
    'prediction_max_det': PRED_MAX_DET,
    'augmentation': {
        'mosaic': 0.0,
        'close_mosaic': 0,
        'copy_paste': 0.0,
        'degrees': 2,
        'translate': 0.02,
        'scale': 0.10,
        'fliplr': 0.5,
        'mixup': 0.0,
        'hsv_h': 0.01,
        'hsv_s': 0.25,
        'hsv_v': 0.20,
        'erasing': 0.0,
        'overlap_mask': False,
        'mask_ratio': 2,
        'cls': 1.0,
    },
}
(RUN_DIR / 'yolo_live_training_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
display(Markdown('## Final files'))
for key, value in summary.items():
    print(f'{key}: {value}')
print('Available plots:', [p.name for p in RUN_DIR.glob('*.png')])

In [ ]:
# Optional qualitative check after training: generate predictions on validation tiles.
from ultralytics import YOLO

assert best_pt.exists(), best_pt
model = YOLO(str(best_pt))
pred = model.predict(
    source=str(YOLO_DATASET / 'images/val'),
    imgsz=IMGSZ,
    conf=PRED_CONF,
    iou=PRED_IOU,
    max_det=PRED_MAX_DET,
    save=True,
    project=str(OUTPUT_ROOT),
    name=RUN_NAME + '_predict_val',
)
print('Prediction output:', OUTPUT_ROOT / (RUN_NAME + '_predict_val'))